In [1]:
from icosphere import icosphere
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

In [2]:
# make the geodesic map
r_earth_e = 6356752.3 # equatorial radius
r_earth_p = 6378137.0 # polar radius

m_earth = 5.97219 * 10**24 # earth mass kg (assumed uniform)

nu = 10
vertices, faces = icosphere(nu)
spheres = []
for n in range(10):
    
    vertices, faces = icosphere(n+1)
    sphere = {'nu': n+1,
              'vertices': vertices,
              'faces': faces}
    spheres.append(sphere)
    
    # find the vertices that only apply to 

# what is in each of the faces
sphere = spheres[1]
# print(sphere['faces'])
# print(sphere['vertices'])

# find the vertices that cetner the pentagons
for s in range(len(spheres)):
    sphere = spheres[s]
    vertex_cnt = np.zeros((len(sphere['vertices']),1))
    for f in range(len(sphere['faces'])):
        face_pts = sphere['faces'][f]
        vertex_cnt[face_pts[0]] += 1
        vertex_cnt[face_pts[1]] += 1
        vertex_cnt[face_pts[2]] += 1
    
    print(np.where(vertex_cnt==5)[0])
# always the first 12 vertices that are the centers for the pentagons


[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]


In [3]:
import pyvista as pv
import numpy as np
from scipy.spatial.distance import cdist
from collections import defaultdict
from tqdm import tqdm

TERRAIN_WATER    = 0
TERRAIN_LAND     = 1
TERRAIN_BEACH    = 2
TERRAIN_HILLS    = 3
TERRAIN_MOUNTAIN = 4

def rotation_matrix_to_z(v):
    v = v / np.linalg.norm(v)
    z = np.array([0.0, 0.0, 1.0])
    axis = np.cross(v, z)
    norm = np.linalg.norm(axis)
    if norm < 1e-10:
        return np.eye(3)
    axis /= norm
    angle = np.arccos(np.clip(np.dot(v, z), -1, 1))
    c, s = np.cos(angle), np.sin(angle)
    t = 1 - c
    ax, ay, az = axis
    return np.array([
        [t*ax*ax+c,    t*ax*ay-s*az, t*ax*az+s*ay],
        [t*ax*ay+s*az, t*ay*ay+c,   t*ay*az-s*ax],
        [t*ax*az-s*ay, t*ay*az+s*ax, t*az*az+c   ]
    ])

class world_map(object):
    def __init__(self, config):
        self.config         = config
        self.nu             = self.config.get("nu", 5)
        self.r_polar        = self.config.get("polar_radius", 6378137.0)
        self.r_equatorial   = self.config.get("equator_radius", 6356752.3)
        self.water_bias     = self.config.get("water_bias", 0.6)   # prob border cell becomes water
        self.land_fraction  = self.config.get("land_fraction", 0.4) # initial land fraction at nsub=1

        self.make_map(self.nu, self.r_polar, self.r_equatorial)

    # ------------------------------------------------------------------
    # BUILD MAP
    # ------------------------------------------------------------------
    def make_map(self, subdivisions=5, r_polar=1, r_equatorial=1):
        # Start at nsub=1 (12 faces) and grow up to self.nu
        icosphere    = pv.Icosphere(radius=1.0, nsub=1)
        self.points  = icosphere.points.copy()
        faces        = icosphere.faces.reshape((-1, 4))
        self.faces   = faces[:, 1:]

        self.rotate_sphere()

        # Seed land/water randomly at coarsest level
        n_faces       = len(self.faces)
        self.face_type = np.random.choice(
            [1, 0],                          # 1=land, 0=water
            size=n_faces,
            p=[self.land_fraction, 1 - self.land_fraction]
        )

        # Iteratively subdivide up to target
        for level in range(1, self.nu):
            print(f"Generating Level {level}")
            self._subdivide_and_smooth()

        # secondary terrain pass
        self.apply_terrain_detail(
            beach_depth=self.config.get("beach_depth"),
            mountain_depth=self.config.get("mountain_depth"),
            hill_depth=self.config.get("hill_depth")
        )

        self.plane = self.make_plane()

    # ------------------------------------------------------------------
    # SUBDIVISION + SMOOTHING
    # ------------------------------------------------------------------
    def _subdivide_and_smooth(self):
        """
        Split every triangle into 4 children, inherit parent type,
        then apply border smoothing with water bias.
        """
        old_points = self.points
        old_faces  = self.faces
        old_types  = self.face_type

        new_points = list(old_points)
        new_faces  = []
        new_types  = []
        edge_midpoint_cache = {}  # edge tuple -> new point index

        def get_midpoint(i, j):
            key = (min(i, j), max(i, j))
            if key not in edge_midpoint_cache:
                mid = (old_points[i] + old_points[j]) / 2.0
                mid = mid / np.linalg.norm(mid)  # project back onto unit sphere
                edge_midpoint_cache[key] = len(new_points)
                new_points.append(mid)
            return edge_midpoint_cache[key]

        # loop through each of the faces and their three points
        for fi, (a, b, c) in tqdm(enumerate(old_faces),desc="Subdividing Faces"):
            ab = get_midpoint(a, b)
            bc = get_midpoint(b, c)
            ca = get_midpoint(c, a)

            # 4 children inherit parent type
            children = [
                (a,  ab, ca),
                (b,  bc, ab),
                (c,  ca, bc),
                (ab, bc, ca),   # centre child
            ]
            for child in children:
                new_faces.append(child)
                new_types.append(old_types[fi])

        self.points    = np.array(new_points)
        self.faces     = np.array(new_faces)
        self.face_type = np.array(new_types)

        # Smooth borders with water bias
        self._smooth_borders()

    def _smooth_borders(self):
        """
        Build face adjacency, identify border faces, then
        probabilistically flip them — biased toward water.
        """
        neighbors = self._build_face_adjacency()
        new_types = self.face_type.copy()

        for fi in tqdm(range(len(self.faces)),desc="Smoothing Borders"):
            nbrs = neighbors[fi]
            if not nbrs:
                continue
            nbr_types = self.face_type[list(nbrs)]
            # Only act on border faces (at least one neighbor differs)
            if np.all(nbr_types == self.face_type[fi]):
                continue

            # Weighted vote from neighbors
            water_votes = np.sum(nbr_types == 0)
            land_votes  = np.sum(nbr_types == 1)

            # Apply water bias to the probability
            water_weight = water_votes * self.water_bias
            land_weight  = land_votes  * (1 - self.water_bias)
            total        = water_weight + land_weight

            if total == 0:
                continue

            p_land = land_weight / total
            new_types[fi] = np.random.choice([1, 0], p=[p_land, 1 - p_land])

        self.face_type = new_types

    def _build_face_adjacency(self):
        """
        Returns a dict: face_index -> set of neighboring face indices.
        Two faces are neighbors if they share an edge (2 vertices).
        """
        edge_to_faces = defaultdict(set)
        for fi, (a, b, c) in enumerate(self.faces):
            for edge in [(min(a,b), max(a,b)),
                         (min(b,c), max(b,c)),
                         (min(a,c), max(a,c))]:
                edge_to_faces[edge].add(fi)

        neighbors = defaultdict(set)
        for faces_sharing in edge_to_faces.values():
            for fi in faces_sharing:
                neighbors[fi].update(faces_sharing - {fi})
        return neighbors

    # ------------------------------------------------------------------
    # ROTATION
    # ------------------------------------------------------------------
    def rotate_sphere(self, phi=0, theta=0):
        dists    = cdist(self.points, self.points)
        i, j     = np.unravel_index(np.argmax(dists), dists.shape)
        pole_vec = self.points[i]
        R        = rotation_matrix_to_z(pole_vec)
        self.points = (R @ self.points.T).T

    # ------------------------------------------------------------------
    # 2D PROJECTION
    # ------------------------------------------------------------------
    def make_plane(self):
        x, y, z = self.points[:, 0], self.points[:, 1], self.points[:, 2]
        r        = np.sqrt(x**2 + y**2 + z**2)
        theta    = np.arccos(np.clip(z / r, -1, 1))
        phi      = (np.arctan2(y, x) + 2*np.pi) % (2*np.pi)

        self.points_2d = np.column_stack([phi, theta])

        # Cull seam-crossing faces
        valid_faces = []
        valid_types = []
        for fi, tri in enumerate(self.faces):
            phis = phi[tri]
            if (np.max(phis) - np.min(phis)) < np.pi:
                valid_faces.append(tri)
                valid_types.append(self.face_type[fi])

        self.plane_faces      = np.array(valid_faces)
        self.plane_face_types = np.array(valid_types)

        flat_pts   = np.column_stack([phi, theta, np.zeros(len(phi))])
        face_cells = np.hstack([
            np.full((len(valid_faces), 1), 3), valid_faces
        ]).ravel()

        self.flat_mesh = pv.PolyData(flat_pts, face_cells)
        # Attach land/water as cell scalar for coloring
        self.flat_mesh.cell_data["terrain"] = self.plane_face_types

    def _compute_shore_distance(self):
        """
        BFS from all water-adjacent land cells outward.
        Returns an array of integer hop-distances from shore for each face.
        Water cells get distance -1.
        """
        neighbors = self._build_face_adjacency()
        n = len(self.faces)
        distance = np.full(n, -1, dtype=int)

        # Seed BFS: land faces that touch at least one water face
        queue = []
        for fi in range(n):
            if self.face_type[fi] == TERRAIN_LAND:
                nbr_types = [self.face_type[nb] for nb in neighbors[fi]]
                if TERRAIN_WATER in nbr_types:
                    distance[fi] = 0
                    queue.append(fi)

        # BFS outward through land only
        head = 0
        while head < len(queue):
            fi = queue[head]
            head += 1
            for nb in neighbors[fi]:
                if self.face_type[nb] == TERRAIN_LAND and distance[nb] == -1:
                    distance[nb] = distance[fi] + 1
                    if nb not in queue:
                        queue.append(nb)

        return distance

    def apply_terrain_detail(self, beach_depth, mountain_depth, hill_depth):
        """
        Secondary pass over the completed mesh:
        - Land cells within beach_depth hops of water  -> beach
        - Land cells beyond mountain_depth hops of water -> mountain
        - Everything in between stays as land

        Parameters
        ----------
        beach_depth   : int  cells within this many hops of shore become beach
        mountain_depth: int  cells at least this many hops from shore become mountain
        """
        distance = self._compute_shore_distance()

        for fi in range(len(self.faces)):
            if self.face_type[fi] != TERRAIN_LAND:
                continue
            d = distance[fi]
            if d == -1:
                # Landlocked land unreachable from coast — treat as mountain
                self.face_type[fi] = TERRAIN_LAND
            elif d <= beach_depth:
                self.face_type[fi] = TERRAIN_BEACH
            elif (hill_depth <= d) and (d < mountain_depth):
                self.face_type[fi] = TERRAIN_HILLS
            elif mountain_depth <= d:
                self.face_type[fi] = TERRAIN_MOUNTAIN

        # Rebuild the 2D projection with updated types
        self.plane = self.make_plane()

    # ------------------------------------------------------------------
    # PLOTTING
    # ------------------------------------------------------------------
    def plot_sphere(self, static=False):
        face_cells  = np.hstack([
            np.full((len(self.faces), 1), 3), self.faces
        ]).ravel()
        sphere_mesh = pv.PolyData(self.points, face_cells)
        sphere_mesh.cell_data["terrain"] = self.face_type

        pl = pv.Plotter(notebook=static)
        pl.add_mesh(
            sphere_mesh,
            scalars="terrain",
            cmap=["dodgerblue", "green", "sandybrown", "darkgreen", "white"],
            clim=[0, 4],
            show_edges=False,
            show_scalar_bar=False
        )
        pl.add_axes()
        pl.show()

    def plot_map(self, static=False):
        pl = pv.Plotter(notebook=static)
        pl.add_mesh(
            self.flat_mesh,
            scalars="terrain",
            cmap=["dodgerblue", "green", "sandybrown", "darkgreen", "white"],
            clim=[0, 4],
            show_edges=False,
            show_scalar_bar=False
        )
        pl.add_axes()
        pl.view_xy()
        pl.show()

config = {
    "nu": 6,
    "water_bias": 0.5,
    "land_fraction": 0.5,
    "beach_depth": 2,      # cells from shore -> beach
    "hill_depth": 10,    # cells from shore -> hill
    "mountain_depth": 18       # cells from shore -> mountain
}
# make the actual map
map = world_map(config)
print("map created")
map.plot_sphere()
map.plot_map()


Generating Level 1


Subdividing Faces: 80it [00:00, 26871.49it/s]
Smoothing Borders: 100%|██████████| 320/320 [00:00<00:00, 108898.77it/s]


Generating Level 2


Subdividing Faces: 320it [00:00, 162294.71it/s]
Smoothing Borders: 100%|██████████| 1280/1280 [00:00<00:00, 133716.29it/s]


Generating Level 3


...

Generating Level 4


Subdividing Faces: 5120it [00:00, 198706.77it/s]
Smoothing Borders: 100%|██████████| 20480/20480 [00:00<00:00, 208718.99it/s]


Generating Level 5


Subdividing Faces: 20480it [00:00, 202719.98it/s]
Smoothing Borders: 100%|██████████| 81920/81920 [00:00<00:00, 241688.13it/s]


map created
